# Qwen-VL PDF解析

使用阿里云Model Studio的Qwen-VL模型对本地PDF文件的第一页进行解析

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pdf2image import convert_from_path
import base64
import requests
import json

## 将PDF转换为图像

In [ ]:
# PDF文件路径
pdf_path = r"E:\AI\AI大模型课程学习\第10周：多模态大模型\Week10\数电票功能介绍.pdf"

# 将PDF第一页转换为图像
print("正在将PDF转换为图像...")
images = convert_from_path(pdf_path, first_page=1, last_page=1)
image = images[0]

# 显示图像
plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.axis('off')
plt.title('PDF第一页')
plt.show()

## 配置阿里云API

In [ ]:
# 请注意：请将下面的API密钥替换为你的阿里云AccessKey
# 你需要先在阿里云控制台创建AccessKey：https://ram.console.aliyun.com/

# 设置API密钥（请替换为你的密钥）
access_key_id = "YOUR_ACCESS_KEY_ID"
access_key_secret = "YOUR_ACCESS_KEY_SECRET"

# 设置API端点
url = "https://dashscope.cn/api/v1/services/aigc/text-generation/generation"

## 将图像转换为Base64编码

In [ ]:
import io

# 将图像转换为Base64编码
buffer = io.BytesIO()
image.save(buffer, format='PNG')
image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
image_data_url = f"data:image/png;base64,{image_base64}"

## 使用阿里云API调用Qwen-VL

In [ ]:
def call_qwen_vl(image_data_url, prompt):
    """调用阿里云Qwen-VL模型"""
    headers = {
        "Authorization": f"Bearer {access_key_id}:{access_key_secret}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "qwen-vl-plus",
        "input": {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_data_url
                            }
                        },
                        {
                            "type": "text",
                            "text": prompt
                        }
                    ]
                }
            ]
        },
        "parameters": {
            "temperature": 0.7,
            "max_tokens": 2048
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["output"]["text"]
    except Exception as e:
        return f"调用API失败: {str(e)}"


## 详细描述PDF内容

In [ ]:
# 提问：请详细描述这张图片的内容
prompt = "请详细描述这张图片的内容，包括标题、主要内容、关键信息等。"

# 使用Qwen-VL进行解析
print("正在使用Qwen-VL解析PDF内容...")
result = call_qwen_vl(image_data_url, prompt)

# 显示解析结果
print("\n解析结果：")
print(result)

## 简要概括PDF内容

In [ ]:
# 自定义问题
custom_prompt = "这张图片的主要内容是什么？请用简洁的语言概括。"

# 使用Qwen-VL进行解析
custom_result = call_qwen_vl(image_data_url, custom_prompt)

# 显示自定义解析结果
print("\n自定义问题解析结果：")
print(custom_result)

## 提取关键信息

In [ ]:
# 提取关键信息的问题
info_prompt = "请提取这张图片中的关键信息，包括标题、核心功能点、重要数据等。"

# 使用Qwen-VL提取关键信息
info_result = call_qwen_vl(image_data_url, info_prompt)

# 显示提取结果
print("\n关键信息提取结果：")
print(info_result)

## 注意事项

1. 请确保你已经在阿里云控制台创建了AccessKey：https://ram.console.aliyun.com/
2. 请将代码中的 `YOUR_ACCESS_KEY_ID` 和 `YOUR_ACCESS_KEY_SECRET` 替换为你的实际密钥
3. 请确保你的阿里云账号已经开通了Model Studio服务
4. 调用API可能会产生费用，请参考阿里云的定价说明